# OVI: Twin Backbone Cross-Modal Fusion — Training Pipeline Demo

> **Paper:** [OVI: Twin Backbone Cross-Modal Fusion for Audio-Video Generation](https://arxiv.org/abs/2510.01284)  
> **Affiliation:** Character AI + Yale University  
> **Purpose:** This notebook is a **demonstration-only** implementation of the OVI training pipeline. It illustrates the architecture and training stages described in the paper using simplified, runnable code.

---

## Pipeline Overview

OVI is trained in two sequential stages:

| Stage | Name | What it does |
|-------|------|--------------|
| 1 | **Audio Pretraining** | Trains an audio DiT from scratch on speech + SFX |
| 1b | **Audio Fine-tuning** | Fine-tunes audio tower on 5s clips to match video duration |
| 2 | **AV Fusion Training** | Jointly trains both towers + new cross-modal attention layers |

```
Stage 1: Audio Tower (OVI-AUD)
  ┌─────────────────────────┐
  │  Audio Latents (1D VAE) │
  │  Flow Matching Loss     │
  │  T5 Text Conditioning   │
  └─────────────────────────┘
            ↓
Stage 2: Twin Backbone Fusion
  ┌──────────────────────────────────────────┐
  │  Video DiT (Wan2.2 init) ←→ Audio DiT   │
  │  Blockwise Bidirectional Cross-Attention │
  │  Scaled RoPE for temporal alignment      │
  │  Shared T5 conditioning (combined prompt)│
  └──────────────────────────────────────────┘
```

## 0. Imports & Setup

In [ ]:
# Core dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import math
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"Device : {DEVICE}")
print(f"Dtype  : {DTYPE}")

---
## 1. Architecture: Rotary Position Embeddings (RoPE)

OVI uses scaled RoPE to align the temporal positions of audio and video tokens.  
- Video latents: **31 frames**  
- Audio latents: **157 tokens** (16 kHz × 5 s / 512 stride)
- Audio RoPE scale: `31 / 157 ≈ 0.197` → audio tokens attend to video in sync

In [ ]:
def build_rope_freqs(seq_len: int, head_dim: int, base: float = 10000.0,
                     scale: float = 1.0) -> torch.Tensor:
    """Compute RoPE frequency matrix for one temporal axis.
    
    Args:
        seq_len  : number of tokens along this axis
        head_dim : dimensionality of each attention head
        base     : theta base for frequency calculation
        scale    : optional scaling factor (used for audio/video alignment in OVI)
    Returns:
        freqs : (seq_len, head_dim // 2) complex tensor
    """
    half_dim = head_dim // 2
    # θ_i = 1 / (base^(2i/d))
    theta = 1.0 / (base ** (torch.arange(0, half_dim, dtype=torch.float32) / half_dim))
    # Apply temporal scale (key trick from the paper)
    positions = torch.arange(seq_len, dtype=torch.float32) * scale
    # Outer product → (seq_len, half_dim)
    freqs = torch.outer(positions, theta)
    return torch.polar(torch.ones_like(freqs), freqs)  # complex exponential


def apply_rope(x: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to query or key tensor.
    
    Args:
        x     : (..., seq_len, head_dim) real tensor
        freqs : (seq_len, head_dim // 2) complex tensor
    """
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs     = freqs.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, head_dim//2)
    x_rotated = x_complex * freqs
    return torch.view_as_real(x_rotated).reshape(x.shape).to(x.dtype)


# --- Visualise RoPE alignment (paper Figure 2) ---
VIDEO_FRAMES = 31
AUDIO_TOKENS = 157
HEAD_DIM     = 64   # reduced for illustration
AUDIO_SCALE  = VIDEO_FRAMES / AUDIO_TOKENS  # ≈ 0.197

video_freqs       = build_rope_freqs(VIDEO_FRAMES, HEAD_DIM, scale=1.0)
audio_freqs_raw   = build_rope_freqs(AUDIO_TOKENS, HEAD_DIM, scale=1.0)
audio_freqs_scaled= build_rope_freqs(AUDIO_TOKENS, HEAD_DIM, scale=AUDIO_SCALE)

def compute_affinity(q_freqs, k_freqs):
    """Dot-product affinity between RoPE-rotated unit vectors."""
    q = torch.view_as_real(q_freqs).flatten(-2)  # (N, head_dim)
    k = torch.view_as_real(k_freqs).flatten(-2)  # (M, head_dim)
    q = F.normalize(q, dim=-1)
    k = F.normalize(k, dim=-1)
    return (q @ k.T).numpy()

aff_unscaled = compute_affinity(video_freqs, audio_freqs_raw)
aff_scaled   = compute_affinity(video_freqs, audio_freqs_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mat, title in zip(axes,
                           [aff_unscaled, aff_scaled],
                           ["Default (Unscaled) RoPE Affinity",
                            f"Scaled RoPE Affinity (scale={AUDIO_SCALE:.3f})"]):
    im = ax.imshow(mat, aspect='auto', cmap='inferno',
                   extent=[0, AUDIO_TOKENS, VIDEO_FRAMES, 0])
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Audio token position")
    ax.set_ylabel("Video frame position")
    plt.colorbar(im, ax=ax)

plt.suptitle("Cross-Modal RoPE Affinity (Paper Figure 2)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("rope_affinity.png", dpi=120)
plt.show()
print(f"\nAudio RoPE scale: {AUDIO_SCALE:.4f}")
print("With scaling, the diagonal aligns → accurate temporal correspondence")

---
## 2. Architecture: Minimal DiT Block

Each transformer block in OVI's twin backbone contains:
1. **Self-Attention** (with RoPE)
2. **Cross-Attention to Text** (T5 conditioning)
3. **Cross-Attention to Other Modality** (bidirectional AV fusion — new in OVI)
4. **FFN** (frozen during fusion stage)

In [ ]:
@dataclass
class OVIConfig:
    """Hyperparameters matching Table 1 of the OVI paper (scaled down for demo)."""
    # Full-scale values from Table 1: dim=3072, ffn=14336, heads=24, head_dim=128, blocks=30
    dim:       int = 256    # 3072 in full model
    ffn_dim:   int = 512    # 14336 in full model
    n_heads:   int = 4      # 24 in full model
    head_dim:  int = 64     # 128 in full model
    n_blocks:  int = 4      # 30 in full model
    text_dim:  int = 128    # T5 hidden dim (4096 in full UMT5-XXL)
    dropout:   float = 0.0


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * x / norm


class MultiHeadAttention(nn.Module):
    """Multi-head attention with optional RoPE and cross-attention support."""

    def __init__(self, dim: int, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        inner = n_heads * head_dim
        self.q = nn.Linear(dim, inner, bias=False)
        self.k = nn.Linear(dim, inner, bias=False)
        self.v = nn.Linear(dim, inner, bias=False)
        self.out = nn.Linear(inner, dim, bias=False)
        self.scale = head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                context: Optional[torch.Tensor] = None,
                rope_freqs: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, N, _ = x.shape
        ctx = context if context is not None else x
        M = ctx.shape[1]

        q = self.q(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k(ctx).view(B, M, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v(ctx).view(B, M, self.n_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to Q and K (self-attention only)
        if rope_freqs is not None and context is None:
            # q, k: (B, heads, N, head_dim)
            q = apply_rope(q.transpose(1, 2), rope_freqs).transpose(1, 2)
            k = apply_rope(k.transpose(1, 2), rope_freqs).transpose(1, 2)

        attn = torch.softmax(q @ k.transpose(-2, -1) * self.scale, dim=-1)
        out  = (attn @ v).transpose(1, 2).reshape(B, N, -1)
        return self.out(out)


class AdaLN(nn.Module):
    """Adaptive LayerNorm modulation conditioned on timestep embedding."""

    def __init__(self, dim: int):
        super().__init__()
        self.norm = RMSNorm(dim)
        self.proj = nn.Linear(dim, 2 * dim)  # scale + shift

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        scale, shift = self.proj(t_emb).unsqueeze(1).chunk(2, dim=-1)
        return self.norm(x) * (1 + scale) + shift


class OVIDiTBlock(nn.Module):
    """Single transformer block with:
      1) Self-Attention (+ RoPE)
      2) Text Cross-Attention (T5 conditioning)
      3) Cross-Modal Attention (AV fusion — bidirectional)
      4) FFN
    """

    def __init__(self, cfg: OVIConfig):
        super().__init__()
        self.self_attn    = MultiHeadAttention(cfg.dim, cfg.n_heads, cfg.head_dim)
        self.text_attn    = MultiHeadAttention(cfg.dim, cfg.n_heads, cfg.head_dim)
        self.cross_attn   = MultiHeadAttention(cfg.dim, cfg.n_heads, cfg.head_dim)

        self.norm_sa  = AdaLN(cfg.dim)
        self.norm_ta  = RMSNorm(cfg.dim)
        self.norm_ca  = RMSNorm(cfg.dim)
        self.norm_ffn = AdaLN(cfg.dim)

        self.ffn = nn.Sequential(
            nn.Linear(cfg.dim, cfg.ffn_dim),
            nn.GELU(),
            nn.Linear(cfg.ffn_dim, cfg.dim),
        )

    def forward(self, x: torch.Tensor,
                t_emb: torch.Tensor,
                text_ctx: torch.Tensor,
                other_modal: Optional[torch.Tensor] = None,
                rope_freqs: Optional[torch.Tensor] = None) -> torch.Tensor:
        # 1. Self-attention
        x = x + self.self_attn(self.norm_sa(x, t_emb), rope_freqs=rope_freqs)
        # 2. Text cross-attention (T5)
        x = x + self.text_attn(self.norm_ta(x), context=text_ctx)
        # 3. Cross-modal attention (AV fusion, new in OVI)
        if other_modal is not None:
            x = x + self.cross_attn(self.norm_ca(x), context=other_modal)
        # 4. FFN
        x = x + self.ffn(self.norm_ffn(x, t_emb))
        return x


print("DiT block defined.")
cfg = OVIConfig()
block = OVIDiTBlock(cfg)
n_params = sum(p.numel() for p in block.parameters())
print(f"Parameters per block (demo scale): {n_params:,}")
# Full-scale equivalent: 30 blocks × ~373M per block ≈ 11B total
print(f"Projected full-scale parameters: ~11B (30 blocks × 3072-dim)")

---
## 3. Architecture: Full Twin Backbone (OVI-AUD and OVI-VID)

Both towers share the identical architecture. The **audio tower** is trained from scratch; the **video tower** is initialised from Wan2.2.

In [ ]:
class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep → learnable MLP embedding."""

    def __init__(self, dim: int, freq_dim: int = 256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(freq_dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )
        self.freq_dim = freq_dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.freq_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / half
        )
        emb = t[:, None].float() * freqs[None]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return self.mlp(emb)


class OVITower(nn.Module):
    """One branch of the OVI twin backbone (audio OR video).

    Handles:
      - Patch embedding (linear projection of latent patches)
      - Timestep embedding (AdaLN conditioning)
      - N transformer blocks (self + text + cross-modal attention)
      - Output projection back to latent space
    """

    def __init__(self, cfg: OVIConfig, latent_dim: int, modality: str = "video"):
        super().__init__()
        self.modality = modality

        # Patch embedding: project latent tokens into model dim
        self.patch_embed = nn.Linear(latent_dim, cfg.dim)
        self.t_embed     = TimestepEmbedding(cfg.dim)
        self.blocks      = nn.ModuleList([OVIDiTBlock(cfg) for _ in range(cfg.n_blocks)])
        self.out_norm    = RMSNorm(cfg.dim)
        self.out_proj    = nn.Linear(cfg.dim, latent_dim)

        self._init_weights()

    def _init_weights(self):
        # Zero-init output projection (standard DiT initialisation)
        nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)

    def forward(self, latents: torch.Tensor,
                t: torch.Tensor,
                text_ctx: torch.Tensor,
                other_latents: Optional[torch.Tensor] = None,
                rope_freqs: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            latents       : (B, N, latent_dim) noisy latent tokens
            t             : (B,) timestep scalars ∈ [0, 1]
            text_ctx      : (B, T, dim) T5 text context
            other_latents : (B, M, dim) tokens from the other modality
            rope_freqs    : precomputed RoPE frequencies
        Returns:
            velocity prediction (same shape as latents)
        """
        x     = self.patch_embed(latents)                     # (B, N, dim)
        t_emb = self.t_embed(t)                               # (B, dim)

        # Project other-modality tokens to this model's dim if needed
        other = other_latents  # already in model dim after patch_embed in FusionModel

        for block in self.blocks:
            x = block(x, t_emb, text_ctx, other_modal=other, rope_freqs=rope_freqs)

        return self.out_proj(self.out_norm(x))                # (B, N, latent_dim)


print("OVITower defined.")

VIDEO_LATENT_DIM = 16   # Wan2.2 video latent channels (16 in full model)
AUDIO_LATENT_DIM = 8    # MMAudio 1D VAE latent dim

vid_tower = OVITower(cfg, latent_dim=VIDEO_LATENT_DIM, modality="video")
aud_tower = OVITower(cfg, latent_dim=AUDIO_LATENT_DIM, modality="audio")

vid_params = sum(p.numel() for p in vid_tower.parameters())
aud_params = sum(p.numel() for p in aud_tower.parameters())
print(f"Video tower params (demo): {vid_params:,}")
print(f"Audio tower params (demo): {aud_params:,}")
print(f"Total twin backbone (demo): {vid_params + aud_params:,}")
print(f"Full-scale equivalent: ~11B (5B per branch × 2)")

---
## 4. Architecture: OVI Fusion Model

The `OVIFusionModel` wraps both towers and enables blockwise bidirectional cross-modal exchange.

In [ ]:
class OVIFusionModel(nn.Module):
    """Twin backbone with blockwise bidirectional cross-modal fusion.

    Mirrors the architecture in Section 4.1 of the paper.
    At each block:
      - video block receives audio hidden states as cross-modal context
      - audio block receives video hidden states as cross-modal context
    """

    def __init__(self, cfg: OVIConfig,
                 video_latent_dim: int, audio_latent_dim: int,
                 audio_rope_scale: float = 31 / 157):
        super().__init__()
        self.video_tower = OVITower(cfg, video_latent_dim, modality="video")
        self.audio_tower = OVITower(cfg, audio_latent_dim, modality="audio")
        self.audio_rope_scale = audio_rope_scale
        self.cfg = cfg

    def forward(self,
                video_latents: torch.Tensor,  # (B, Nv, Cv)
                audio_latents: torch.Tensor,  # (B, Na, Ca)
                t: torch.Tensor,               # (B,)  shared timestep
                text_ctx: torch.Tensor,        # (B, T, dim)  shared T5 context
                ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            video_velocity : (B, Nv, Cv)
            audio_velocity : (B, Na, Ca)
        """
        B, Nv, Cv = video_latents.shape
        B, Na, Ca = audio_latents.shape

        # Build RoPE frequencies
        vid_rope = build_rope_freqs(Nv, self.cfg.head_dim).to(video_latents.device)
        aud_rope = build_rope_freqs(Na, self.cfg.head_dim,
                                    scale=self.audio_rope_scale).to(audio_latents.device)

        # Embed patches
        v = self.video_tower.patch_embed(video_latents)
        a = self.audio_tower.patch_embed(audio_latents)
        v_t = self.video_tower.t_embed(t)
        a_t = self.audio_tower.t_embed(t)  # shared t, independent embeddings

        # Blockwise bidirectional fusion
        for v_block, a_block in zip(self.video_tower.blocks, self.audio_tower.blocks):
            # Each modality uses the OTHER modality's current hidden state as context
            v_new = v_block(v, v_t, text_ctx, other_modal=a, rope_freqs=vid_rope)
            a_new = a_block(a, a_t, text_ctx, other_modal=v, rope_freqs=aud_rope)
            v, a  = v_new, a_new  # update simultaneously (not sequentially)

        video_vel = self.video_tower.out_proj(self.video_tower.out_norm(v))
        audio_vel = self.audio_tower.out_proj(self.audio_tower.out_norm(a))
        return video_vel, audio_vel


model = OVIFusionModel(cfg, VIDEO_LATENT_DIM, AUDIO_LATENT_DIM)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params  : {total:,}")
print(f"Trainable     : {trainable:,}")

# Quick forward pass sanity check
B, Nv, Na = 2, 31, 157
T = 8   # text tokens
v_in  = torch.randn(B, Nv, VIDEO_LATENT_DIM)
a_in  = torch.randn(B, Na, AUDIO_LATENT_DIM)
t_in  = torch.rand(B)
ctx   = torch.randn(B, T, cfg.dim)  # normally from T5; here random for demo

with torch.no_grad():
    v_vel, a_vel = model(v_in, a_in, t_in, ctx)

print(f"\nForward pass OK")
print(f"  video velocity shape : {v_vel.shape}  (expected {v_in.shape})")
print(f"  audio velocity shape : {a_vel.shape}  (expected {a_in.shape})")

---
## 5. Training Objective: Flow Matching

OVI uses **Flow Matching** (Lipman et al., 2022) instead of standard DDPM noise prediction.

**Interpolant:**  
$$z_t = (1-t)\, z_0 + t\, z_1, \quad t \sim \mathcal{U}[0,1]$$

where $z_1$ is the clean data and $z_0 \sim \mathcal{N}(0, I)$ is Gaussian noise.

**Velocity target:**  
$$v^* = z_1 - z_0$$

**Loss:**  
$$\mathcal{L}_{FM} = \mathbb{E}_{t,z_1,z_0}\left[\| v_\theta(z_t, t, c_{\text{text}}) - (z_1 - z_0) \|^2_2\right]$$

**Combined AV loss (Stage 2):**  
$$\mathcal{L}_{\text{total}} = \lambda_v \mathcal{L}^v_{FM} + \lambda_a \mathcal{L}^a_{FM}, \quad \lambda_v=0.85,\; \lambda_a=0.15$$

In [ ]:
def flow_matching_loss(model_out: torch.Tensor,
                        z_clean: torch.Tensor,
                        z_noise: torch.Tensor) -> torch.Tensor:
    """Compute the flow matching loss.

    Args:
        model_out : velocity prediction from the model
        z_clean   : clean data z_1
        z_noise   : sampled noise z_0
    Returns:
        scalar MSE loss
    """
    target = z_clean - z_noise     # ground-truth velocity
    return F.mse_loss(model_out, target)


def sample_flow_matching(
    z_clean: torch.Tensor,
    z_noise: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Sample a random interpolation time t and form the noisy latent z_t.

    Returns:
        z_t : (B, ...) interpolated latent
        t   : (B,)     interpolation time
    """
    B = z_clean.shape[0]
    t = torch.rand(B, device=z_clean.device)                       # t ~ U[0,1]
    t_b = t.view(B, *([1] * (z_clean.dim() - 1)))                  # broadcast shape
    z_t = (1 - t_b) * z_noise + t_b * z_clean                     # linear interpolant
    return z_t, t


# --- Demonstrate flow matching interpolation ---
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
z1 = torch.randn(1, 64, 8)   # mock clean latent
z0 = torch.randn(1, 64, 8)   # mock noise

for ax, t_val in zip(axes, [0.0, 0.25, 0.5, 0.75, 1.0]):
    z_t = (1 - t_val) * z0 + t_val * z1
    ax.imshow(z_t[0].numpy(), aspect='auto', cmap='RdBu', vmin=-3, vmax=3)
    ax.set_title(f"t = {t_val}")
    ax.axis('off')

plt.suptitle("Flow Matching Interpolation: Noise → Clean", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("flow_matching.png", dpi=120)
plt.show()
print("At t=0: pure noise (z_0)")
print("At t=1: clean data (z_1)")
print("Model learns to predict the velocity v* = z_1 - z_0")

---
## 6. Stage 1: Audio-Only Pretraining (OVI-AUD)

The audio tower is trained **from scratch** on hundreds of thousands of hours of speech and sound effects **before** any video data is seen.

**Key details from the paper:**
- 50k steps, batch size 2880, lr = 1×10⁻⁴
- Up to 12-second waveforms in pretraining
- AdamW (β₁=0.9, β₂=0.999, ε=10⁻⁸)
- Scaled RoPE applied at all stages to avoid re-adaptation

In [ ]:
class AudioOnlyDataset(Dataset):
    """Mock dataset for the audio-only pretraining stage.

    In the real pipeline this loads audio latents encoded by the MMAudio 1D VAE,
    along with T5 text embeddings of the audio caption + transcript.
    """

    def __init__(self, n_samples: int = 512,
                 audio_tokens: int = 157,
                 audio_latent_dim: int = 8,
                 text_tokens: int = 64,
                 text_dim: int = 128):
        self.n = n_samples
        self.audio_tokens = audio_tokens
        self.audio_latent_dim = audio_latent_dim
        self.text_tokens = text_tokens
        self.text_dim = text_dim

    def __len__(self): return self.n

    def __getitem__(self, idx):
        # In practice: load from disk / encode with VAE + T5
        audio_latent = torch.randn(self.audio_tokens, self.audio_latent_dim)
        text_ctx     = torch.randn(self.text_tokens, self.text_dim)
        return {"audio": audio_latent, "text": text_ctx}


def train_audio_pretrain(n_steps: int = 50, log_every: int = 10):
    """Stage 1 audio pretraining loop (scaled-down demonstration)."""

    cfg_audio = OVIConfig(dim=128, ffn_dim=256, n_heads=4, head_dim=32, n_blocks=2,
                          text_dim=128)
    audio_tower = OVITower(cfg_audio, latent_dim=AUDIO_LATENT_DIM, modality="audio")
    audio_tower.to(DEVICE)

    dataset = AudioOnlyDataset(n_samples=256, audio_tokens=157,
                               audio_latent_dim=AUDIO_LATENT_DIM, text_dim=128)
    loader  = DataLoader(dataset, batch_size=8, shuffle=True)

    # AdamW with paper's hyperparameters
    optimizer = torch.optim.AdamW(
        audio_tower.parameters(),
        lr=1e-4,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-2
    )

    rope_freqs = build_rope_freqs(157, cfg_audio.head_dim,
                                  scale=31 / 157).to(DEVICE)

    loss_history = []
    step = 0

    for batch in loader:
        if step >= n_steps:
            break

        z1   = batch["audio"].to(DEVICE)           # clean audio latents
        text = batch["text"].to(DEVICE)             # T5 context
        B    = z1.shape[0]

        z0      = torch.randn_like(z1)              # noise
        z_t, t  = sample_flow_matching(z1, z0)      # interpolate

        # Forward pass (audio-only, no cross-modal context)
        v_pred = audio_tower(z_t, t, text, rope_freqs=rope_freqs)
        loss   = flow_matching_loss(v_pred, z1, z0)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(audio_tower.parameters(), 1.0)
        optimizer.step()

        loss_history.append(loss.item())

        if step % log_every == 0:
            print(f"  [Audio Pretrain] step {step:4d} | loss {loss.item():.4f}")
        step += 1

    return audio_tower, loss_history


print("Stage 1: Audio Pretraining")
print("=" * 50)
audio_tower, audio_losses = train_audio_pretrain(n_steps=50, log_every=10)

plt.figure(figsize=(8, 3))
plt.plot(audio_losses, color='steelblue', linewidth=2)
plt.xlabel("Step")
plt.ylabel("Flow Matching Loss")
plt.title("Stage 1: Audio Pretraining Loss Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("audio_pretrain_loss.png", dpi=120)
plt.show()

---
## 7. Stage 2: Audio-Video Fusion Training

The video tower (Wan2.2 init) and pretrained audio tower are combined.  
**Frozen:** All FFN layers (reduces trainable params from 11B → 5.7B)  
**Trained:** Self-attention, text cross-attention, and newly-initialised AV cross-attention

**Key details:**
- 40k steps, batch size 768, lr = 5×10⁻⁵
- AdamW (β₁=0.9, β₂=0.95, ε=10⁻⁸)
- λᵥ=0.85, λₐ=0.15 (video loss weighted higher)
- Shared timestep t across both modalities

In [ ]:
class AVDataset(Dataset):
    """Mock paired audio-video dataset for fusion training.

    In the real pipeline:
      - Video latents come from the Wan2.2 3D VAE (31 × H/8 × W/8 × 16)
      - Audio latents come from the MMAudio 1D VAE (157 × 8)
      - Text context is T5 embedding of the combined prompt:
          "[visual description] <S>[speech]<E> <AUDCAP>[audio description]<ENDAUDCAP>"
    """

    def __init__(self, n_samples: int = 512, text_dim: int = 128):
        self.n = n_samples
        self.text_dim = text_dim

    def __len__(self): return self.n

    def __getitem__(self, idx):
        # Combined-prompt T5 embedding (single encoder for both towers)
        text_ctx     = torch.randn(64, self.text_dim)
        video_latent = torch.randn(31, VIDEO_LATENT_DIM)
        audio_latent = torch.randn(157, AUDIO_LATENT_DIM)
        return {"video": video_latent, "audio": audio_latent, "text": text_ctx}


def freeze_ffns(model: OVIFusionModel):
    """Freeze all FFN parameters in both towers (paper Section 4.2.2)."""
    frozen = 0
    for name, param in model.named_parameters():
        if ".ffn." in name:
            param.requires_grad = False
            frozen += param.numel()
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Frozen {frozen:,} FFN params ({frozen/total*100:.1f}% of total)")
    print(f"  Trainable: {trainable:,} / {total:,}")


def train_av_fusion(n_steps: int = 50, log_every: int = 10,
                    lambda_v: float = 0.85, lambda_a: float = 0.15):
    """Stage 2 audio-video fusion training loop (scaled-down demonstration)."""

    fusion_model = OVIFusionModel(
        cfg=OVIConfig(dim=128, ffn_dim=256, n_heads=4, head_dim=32, n_blocks=2, text_dim=128),
        video_latent_dim=VIDEO_LATENT_DIM,
        audio_latent_dim=AUDIO_LATENT_DIM,
    ).to(DEVICE)

    print("Applying FFN freezing...")
    freeze_ffns(fusion_model)

    dataset = AVDataset(n_samples=256, text_dim=128)
    loader  = DataLoader(dataset, batch_size=4, shuffle=True)

    # Only optimise trainable (non-frozen) params
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, fusion_model.parameters()),
        lr=5e-5,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=1e-2
    )

    loss_history   = []
    loss_v_history = []
    loss_a_history = []
    step = 0

    for batch in loader:
        if step >= n_steps:
            break

        z1v  = batch["video"].to(DEVICE)
        z1a  = batch["audio"].to(DEVICE)
        text = batch["text"].to(DEVICE)
        B    = z1v.shape[0]

        # Independent noise, shared timestep (Section 4.2.2)
        z0v = torch.randn_like(z1v)
        z0a = torch.randn_like(z1a)
        t   = torch.rand(B, device=DEVICE)   # ← shared t
        t_b_v = t.view(B, 1, 1)
        t_b_a = t.view(B, 1, 1)
        z_tv = (1 - t_b_v) * z0v + t_b_v * z1v
        z_ta = (1 - t_b_a) * z0a + t_b_a * z1a

        # Forward through fusion model
        v_vel, a_vel = fusion_model(z_tv, z_ta, t, text)

        # Per-modality FM losses
        loss_v = flow_matching_loss(v_vel, z1v, z0v)
        loss_a = flow_matching_loss(a_vel, z1a, z0a)

        # Weighted total loss (paper equation)
        loss = lambda_v * loss_v + lambda_a * loss_a

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, fusion_model.parameters()), 1.0
        )
        optimizer.step()

        loss_history.append(loss.item())
        loss_v_history.append(loss_v.item())
        loss_a_history.append(loss_a.item())

        if step % log_every == 0:
            print(f"  [AV Fusion] step {step:3d} | total {loss.item():.4f} "
                  f"| video {loss_v.item():.4f} | audio {loss_a.item():.4f}")
        step += 1

    return fusion_model, loss_history, loss_v_history, loss_a_history


print("Stage 2: Audio-Video Fusion Training")
print("=" * 50)
fusion_model, total_losses, v_losses, a_losses = train_av_fusion(n_steps=50, log_every=10)

In [ ]:
# Visualise Stage 2 loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(total_losses, label='Total (λ_v·L_v + λ_a·L_a)', color='purple', lw=2)
axes[0].set_title("Stage 2: Combined AV Loss", fontsize=12)
axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(v_losses, label='Video FM Loss (λ_v = 0.85)', color='steelblue', lw=2)
axes[1].plot(a_losses, label='Audio FM Loss (λ_a = 0.15)', color='coral',     lw=2)
axes[1].set_title("Stage 2: Per-Modality Losses", fontsize=12)
axes[1].set_xlabel("Step"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("OVI Fusion Training Loss Curves", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("fusion_loss.png", dpi=120)
plt.show()

---
## 8. Inference: UniPC ODE Solver

At inference, OVI uses the **UniPC** (Unified Predictor-Corrector) solver, which the authors found more stable than Euler.

Both branches share the **same timestep schedule** and are integrated simultaneously with a single ODE solve.

Classifier-Free Guidance (CFG) is applied per modality:
$$\hat{v} = v_{\text{neg}} + w \cdot (v_{\text{pos}} - v_{\text{neg}})$$

In [ ]:
def euler_solver(model: OVIFusionModel,
                 text_ctx: torch.Tensor,
                 text_ctx_neg: torch.Tensor,
                 video_shape: Tuple,
                 audio_shape: Tuple,
                 n_steps: int = 20,
                 cfg_video: float = 4.0,
                 cfg_audio: float = 3.0,
                 seed: int = 42) -> Tuple[torch.Tensor, torch.Tensor]:
    """Euler ODE solver for OVI inference with classifier-free guidance.

    In production, OVI uses UniPC for better stability.
    This Euler implementation demonstrates the inference algorithm.

    Args:
        model       : trained OVIFusionModel
        text_ctx    : (B, T, dim) positive T5 context
        text_ctx_neg: (B, T, dim) negative T5 context (empty prompt)
        video_shape : (B, Nv, Cv)
        audio_shape : (B, Na, Ca)
        n_steps     : number of denoising steps (paper uses 50)
        cfg_video   : video CFG scale (default 4.0)
        cfg_audio   : audio CFG scale (default 3.0)
    Returns:
        video_pred : (B, Nv, Cv)  final video latent
        audio_pred : (B, Na, Ca)  final audio latent
    """
    torch.manual_seed(seed)
    model.eval()

    # Start from pure noise at t=0
    z_v = torch.randn(*video_shape, device=DEVICE)
    z_a = torch.randn(*audio_shape, device=DEVICE)

    # Uniform time steps from 0 → 1
    timesteps = torch.linspace(0, 1 - 1/n_steps, n_steps, device=DEVICE)
    dt = 1.0 / n_steps

    with torch.no_grad():
        for t_scalar in timesteps:
            t = t_scalar.expand(video_shape[0])

            # Conditional (positive) prediction
            v_pos, a_pos = model(z_v, z_a, t, text_ctx)
            # Unconditional (negative) prediction
            v_neg, a_neg = model(z_v, z_a, t, text_ctx_neg)

            # Classifier-free guidance
            v_cfg = v_neg + cfg_video * (v_pos - v_neg)
            a_cfg = a_neg + cfg_audio * (a_pos - a_neg)

            # Euler step: z_{t+dt} = z_t + dt * v
            z_v = z_v + dt * v_cfg
            z_a = z_a + dt * a_cfg

    return z_v, z_a


# Demo inference
B = 1
text_pos = torch.randn(B, 64, cfg.dim, device=DEVICE)
text_neg = torch.zeros(B, 64, cfg.dim, device=DEVICE)  # empty prompt

print("Running inference (Euler solver, 20 steps)...")
vid_out, aud_out = euler_solver(
    fusion_model, text_pos, text_neg,
    video_shape=(B, 31, VIDEO_LATENT_DIM),
    audio_shape=(B, 157, AUDIO_LATENT_DIM),
    n_steps=20, cfg_video=4.0, cfg_audio=3.0
)
print(f"Generated video latent : {vid_out.shape}")
print(f"Generated audio latent : {aud_out.shape}")
print("\n→ In production: decode video latents with Wan2.2 3D VAE")
print("→ In production: decode audio latents with MMAudio 1D VAE → mel → BigVGAN waveform")

---
## 9. Data Pipeline: Prompt Formatting

OVI uses a **single combined prompt** fed to a shared T5 encoder. The format varies by model variant:

| Resolution | Format |
|-----------|--------|
| 720×720 | `[visual] <S>[speech]<E> <AUDCAP>[audio desc]<ENDAUDCAP>` |
| 960×960 | `[visual] <S>[speech]<E> Audio: [audio desc]` |

In [ ]:
def format_ovi_prompt_720(
    visual_description: str,
    speech_transcript: Optional[str] = None,
    audio_description: Optional[str] = None
) -> str:
    """Format OVI combined prompt for 720×720 model."""
    prompt = visual_description.strip()
    if speech_transcript:
        prompt += f" <S>{speech_transcript.strip()}<E>"
    if audio_description:
        prompt += f" <AUDCAP>{audio_description.strip()}<ENDAUDCAP>"
    return prompt


def format_ovi_prompt_960(
    visual_description: str,
    speech_transcript: Optional[str] = None,
    audio_description: Optional[str] = None
) -> str:
    """Format OVI combined prompt for 960×960 model."""
    prompt = visual_description.strip()
    if speech_transcript:
        prompt += f" <S>{speech_transcript.strip()}<E>"
    if audio_description:
        prompt += f" Audio: {audio_description.strip()}"
    return prompt


# Example prompts from the paper's domain
examples = [
    {
        "visual": "A woman sits at a grand piano in a sunlit concert hall, her fingers moving gracefully across the keys.",
        "speech": "Music fills the air as she plays.",
        "audio":  "A female pianist performs a gentle classical piece, warm resonant tones, soft reverb in the hall."
    },
    {
        "visual": "Two hikers stand at a mountain summit overlooking a vast foggy valley at sunrise.",
        "speech": "We made it! The view is incredible.",
        "audio":  "Two excited voices, light wind, distant bird calls, footsteps on gravel."
    },
    {
        "visual": "A rocket launches from a seaside pad, surrounded by massive exhaust clouds.",
        "speech": None,
        "audio":  "Thunderous rocket engine roar, deep bass rumble, crackling fire, crowd cheering in the distance."
    }
]

print("OVI Combined Prompt Examples")
print("=" * 60)
for i, ex in enumerate(examples, 1):
    p720 = format_ovi_prompt_720(ex["visual"], ex["speech"], ex["audio"])
    p960 = format_ovi_prompt_960(ex["visual"], ex["speech"], ex["audio"])
    print(f"\nExample {i}:")
    print(f"  720×720 prompt: {p720[:120]}..." if len(p720) > 120 else f"  720×720: {p720}")
    print(f"  960×960 prompt: {p960[:120]}..." if len(p960) > 120 else f"  960×960: {p960}")

---
## 10. Training Summary & Full Pipeline Diagram

In [ ]:
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)

# Stage 1 loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(audio_losses, color='steelblue', lw=2)
ax1.set_title("Stage 1: Audio Pretrain Loss")
ax1.set_xlabel("Step"); ax1.set_ylabel("FM Loss")
ax1.grid(True, alpha=0.3)

# Stage 2 loss
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(v_losses, label='Video', color='steelblue', lw=2)
ax2.plot(a_losses, label='Audio', color='coral',     lw=2)
ax2.set_title("Stage 2: Fusion Losses")
ax2.set_xlabel("Step"); ax2.set_ylabel("FM Loss")
ax2.legend(); ax2.grid(True, alpha=0.3)

# RoPE alignment
ax3 = fig.add_subplot(gs[0, 2])
ax3.imshow(aff_scaled, aspect='auto', cmap='inferno',
           extent=[0, AUDIO_TOKENS, VIDEO_FRAMES, 0])
ax3.set_title("Aligned RoPE Affinity")
ax3.set_xlabel("Audio tokens"); ax3.set_ylabel("Video frames")

# Architecture diagram (text)
ax4 = fig.add_subplot(gs[1, :])
ax4.axis('off')
diagram = """
OVI TRAINING PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  STAGE 1 — Audio Pretraining (50k steps, BS=2880, LR=1e-4)         STAGE 2 — AV Fusion (40k steps, BS=768, LR=5e-5)

  Raw Audio (speech + SFX)                                           Paired AV corpus (millions of videos)
       │                                                                    │
  MMAudio 1D VAE encode                                       Wan2.2 3D VAE encode (video)  +  MMAudio VAE (audio)
       │                                                                    │
  Flow Matching Loss                                               Combined T5 Prompt Embedding (shared)
  L_a = E[||v_θ(z_t, t, c) - (z1-z0)||²]                                 │
       │                                              ┌─────────────────────────────────────┐
  OVI-AUD (audio tower trained from scratch)          │ Video DiT ←─── cross-attn ────→ Audio DiT │
  Identical arch to Wan2.2 5B                         │ (Wan2.2 init)  (blockwise, bidir)  (OVI-AUD)│
                                                      │         Scaled RoPE alignment              │
                                                      │    L_total = 0.85·L_v + 0.15·L_a           │
                                                      └─────────────────────────────────────┘
                                                                    FFNs FROZEN
"""
ax4.text(0.02, 0.95, diagram, transform=ax4.transAxes,
         fontsize=8.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))

fig.suptitle("OVI: Twin Backbone Cross-Modal Fusion — Training Pipeline Overview",
             fontsize=13, fontweight='bold')
plt.savefig("ovi_pipeline_summary.png", dpi=120, bbox_inches='tight')
plt.show()
print("Pipeline summary saved to ovi_pipeline_summary.png")

---
## 11. Ablation: CLAP vs. Unified T5 Conditioning

The paper (Section 5.5) shows that using a **single T5** for both speech transcript and audio description outperforms a **dual-encoder** approach (T5 for speech + CLAP for sound effects).

Reproducing Table 3 values for reference:

In [ ]:
import pandas as pd

ablation = pd.DataFrame({
    "Variant":    ["OVI with CLAP (T5 + CLAP dual encoder)", "OVI (unified T5 only)"],
    "FDPANNs ↓":  [20.78, 18.03],
    "FDVGG ↓":    [7.13,  5.02],
    "IS ↑":       [8.34,  11.20],
    "CLAP ↑":     [0.190, 0.224],
    "WER ↓":      [0.033, 0.035],
})

print("Table 3 — Ablation Study: Audio Tower Text Conditioning")
print(ablation.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = [("FDPANNs ↓", False), ("IS ↑", True), ("CLAP ↑", True)]
colors = ['salmon', 'steelblue']

for ax, (metric, higher_is_better) in zip(axes, metrics):
    bars = ax.bar(ablation["Variant"], ablation[metric], color=colors, edgecolor='black', width=0.5)
    ax.set_title(metric, fontsize=12)
    ax.set_xticks(range(len(ablation)))
    ax.set_xticklabels(["+ CLAP", "Unified T5"], fontsize=10)
    ax.grid(True, axis='y', alpha=0.3)
    winner = 1 if higher_is_better else 0
    bars[winner].set_edgecolor('green')
    bars[winner].set_linewidth(2.5)

plt.suptitle("Ablation: Unified T5 vs CLAP+T5 Dual Encoder",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("ablation.png", dpi=120)
plt.show()
print("\nConclusion: Unified T5 wins on perceptual quality (FD, IS, CLAP) with comparable WER.")
print("Reason: Single embedding enables better integration of speech + SFX into a coherent stream.")

---
## Summary

This notebook demonstrated the full OVI training pipeline:

| Component | Key Design Choice | Outcome |
|-----------|------------------|---------|
| **Architecture** | Symmetric twin DiT (5B + 5B = 11B) | Same latent dim → no projections needed |
| **RoPE Alignment** | Scale audio RoPE by 31/157 ≈ 0.197 | Diagonal temporal alignment in cross-attention |
| **Stage 1** | Audio tower trained from scratch on speech+SFX | Foundational audio model for both T2A and TTS |
| **Stage 2** | Freeze FFNs, train attention only (5.7B of 11B) | Memory-efficient fusion without forgetting |
| **Loss** | λᵥ=0.85·L_v + λₐ=0.15·L_a, shared timestep | Paired sampling learns AV correspondence implicitly |
| **Conditioning** | Single T5 on combined prompt | Simpler training, stronger cross-modal coherence |
| **Solver** | UniPC (Predictor-Corrector) | More stable than Euler, fewer steps needed |

> For production inference, see the [official OVI repo](https://github.com/character-ai/Ovi) and the [project demo page](https://aaxwaz.github.io/Ovi).